In [1]:
import torch
from torch_geometric.datasets import EllipticBitcoinDataset
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import fbeta_score, average_precision_score
import matplotlib.pyplot as plt
import numpy as np

# 1. Load the Data
print("Downloading/Loading Elliptic Dataset...")
dataset = EllipticBitcoinDataset(root='../data/Elliptic')
data = dataset[0]

# 2. Prepare Tabular Data (Ignoring Graph Edges for Baselines)
X = data.x.numpy()
y = data.y.numpy()

# Use PyG's predefined temporal split
train_idx = data.train_mask.nonzero(as_tuple=True)[0].numpy()
test_idx = data.test_mask.nonzero(as_tuple=True)[0].numpy()

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f"Training nodes: {len(X_train)} | Test nodes: {len(X_test)}")

# 3. Train Baselines
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(n_estimators=100, n_jobs=-1, random_state=42, scale_pos_weight=10)
}

results = {}
print("\nTraining models (this takes a few seconds)...")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    
    # F2 heavily penalizes missing an illicit transaction (false negatives)
    f2 = fbeta_score(y_test, y_pred, beta=2)
    pr_auc = average_precision_score(y_test, y_pred_prob)
    results[name] = {'F2': f2, 'PR-AUC': pr_auc}
    print(f"{name} -> F2: {f2:.4f} | PR-AUC: {pr_auc:.4f}")

C:\Users\Manya Chaubey\elliptic_aml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Manya Chaubey\elliptic_aml\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Downloading/Loading Elliptic Dataset...


Processing...
C:\Users\Manya Chaubey\elliptic_aml\.venv\Lib\site-packages\torch_geometric\datasets\elliptic.py:108: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y = torch.from_numpy(class_df['class'].values)
Done!


Training nodes: 29894 | Test nodes: 16670

Training models (this takes a few seconds)...
Logistic Regression -> F2: 0.5107 | PR-AUC: 0.2771
Random Forest -> F2: 0.7551 | PR-AUC: 0.7951
XGBoost -> F2: 0.7537 | PR-AUC: 0.8000
